[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C04/E04-2_Presse_Brute.ipynb)

# Relevés bruts d'une presse : combler les manques et coder les catégories

## Le contexte

Une presse écrase des cylindres de métal entre deux plateaux, le refoulement. L'effort maximal que la presse doit fournir dépend du diamètre et de la hauteur de la pièce, de sa température, de la vitesse de la traverse et du lubrifiant. Le prévoir permet de choisir la presse et de ne pas la surcharger.

Les relevés viennent de plusieurs postes. Ils sont bruts : diamètres tantôt en millimètres, tantôt en pouces, températures tantôt en degrés Celsius, tantôt en degrés Fahrenheit, valeurs de capteur en défaut, cases vides, lignes enregistrées deux fois. Un modèle qui ingère ces relevés tels quels apprend les défauts de la saisie, pas la mécanique. Il faut nettoyer, combler les manques, coder les catégories, et le faire sans que le test ne fuie vers l'apprentissage.

C'est un problème de régression : la grandeur à prédire est un nombre, en kilonewtons.

Défi : prédire `effort_max_kN` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ nettoie les relevés, comble les cases vides par la médiane et code le lubrifiant, puis ajuste une droite ; il atteint un R² de 0,911 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Presse_Brute.xlsx`, dans le dossier `donnees` du dépôt : 524 relevés, 8 colonnes. Une ligne est un essai de refoulement.

| Colonne | Signification | Unité |
|---|---|---|
| `lubrifiant` | `graisse`, `graphite` ou `sec` | catégorie |
| `diametre` | diamètre de la pièce, dans l'unité de la colonne suivante | mm ou in |
| `unite_diametre` | unité du diamètre : `mm` ou `in` | catégorie |
| `hauteur_mm` | hauteur de la pièce | mm |
| `temperature` | température de la pièce, dans l'unité de la colonne suivante | °C ou °F |
| `unite_temperature` | unité de la température : `C` ou `F` | catégorie |
| `vitesse_mm_s` | vitesse de la traverse | mm/s |
| `effort_max_kN` | cible : effort maximal de la presse | kN |

Ce jeu est simulé : l'effort est calculé par la formule de Siebel du refoulement, puis perturbé ; les défauts de saisie (unités mêlées, valeurs de capteur en défaut, doublons, cases vides) sont ajoutés à dessein. Il enseigne une méthode et ne remplace pas un essai.

### Étape 1 : charger le fichier et repérer les défauts

On lit le classeur, puis on cherche les défauts de saisie : cases vides, lignes en double, unités. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Presse_Brute.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Presse_Brute.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "effort_max_kN"
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides par colonne :")
print(vides[vides > 0].to_string())
n_doublons = int(df.duplicated().sum())
print("lignes identiques à une ligne précédente :", n_doublons)
print(df["unite_diametre"].value_counts().to_dict(), df["unite_temperature"].value_counts().to_dict())
df.head(3)

Lecture du résultat : 524 lignes, 8 colonnes. Deux colonnes ont des cases vides, `hauteur_mm` (19) et `vitesse_mm_s` (19). 24 lignes répètent une ligne précédente. Les unités sont mêlées : une partie des diamètres est en pouces et une partie des températures en degrés Fahrenheit. Les cases vides sont visibles ; les autres défauts demandent de regarder les valeurs.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Presse_Brute.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : regarder les valeurs

On résume chaque mesure, puis on regarde le diamètre et la température dans chaque unité. Des valeurs hors de toute plage physique, comme des codes de capteur en défaut, se repèrent aux extrêmes.

In [ ]:
mesures = ["diametre", "hauteur_mm", "temperature", "vitesse_mm_s"]
print(df[mesures].describe().loc[["count", "min", "50%", "max"]].round(1).to_string())
print()
print(df.groupby("unite_diametre")["diametre"].describe()[["count", "min", "50%", "max"]].round(2).to_string())
print(df.groupby("unite_temperature")["temperature"].describe()[["count", "min", "50%", "max"]].round(0).to_string())
n_defaut_temperature = int((df["temperature"] == 9999).sum())
n_defaut_vitesse = int((df["vitesse_mm_s"] == 999).sum())
n_diametre_nul = int((df["diametre"] == 0).sum())
print("températures à 9999 :", n_defaut_temperature, "; vitesses à 999 :", n_defaut_vitesse, "; diamètres nuls :", n_diametre_nul)

Lecture du résultat : trois défauts apparaissent aux extrêmes : 10 températures valent 9999, 6 vitesses valent 999 et 3 diamètres sont nuls. Ce sont des codes de capteur en défaut, pas des mesures. Les diamètres en pouces se reconnaissent à leur médiane (environ 1,5, contre environ 41 en millimètres) ; les températures en Fahrenheit ont une médiane plus élevée que celles en Celsius. Un modèle qui lirait ces colonnes telles quelles mélangerait des grandeurs qui n'ont pas la même unité.

### Étape 3 : nettoyer ligne à ligne

Le nettoyage ramène les mesures à une même unité (millimètres, degrés Celsius) et remplace les codes de capteur en défaut par une case vide, qui sera comblée plus tard par le pipeline. Ces corrections ne dépendent d'aucune autre ligne : on peut les appliquer à tout le tableau avant le découpage sans que le test ne fuie vers l'apprentissage. Les colonnes d'unité deviennent inutiles et disparaissent. Les lignes en double, elles, se comparent entre elles : on s'en occupe après le découpage.

In [ ]:
# Votre code :
print(propre[mesures].describe().loc[["count", "min", "50%", "max"]].round(1).to_string())

Lecture du résultat : après nettoyage, les diamètres vont de 20,1 à 59,8 mm, les températures de 21 à 350 °C et les vitesses de 5,0 à 49,8 mm/s : des plages plausibles pour un refoulement. Il y a maintenant 3 cases vides en diamètre, 19 en hauteur, 10 en température et 25 en vitesse.

### Étape 4 : séparer l'apprentissage et le test

Un quart des relevés est mis de côté avant tout apprentissage. La graine 42 fixe le tirage sur les 524 lignes du classeur : chacun obtient la même base de test, et les scores se comparent. Les lignes du tableau nettoyé gardent leur position initiale.

In [ ]:
from sklearn.model_selection import train_test_split

X = propre.drop(columns=cible)
y = propre[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "relevés")
print("test :", len(X_test), "relevés")

### Étape 5 : retirer les doublons de l'apprentissage

Une ligne enregistrée deux fois peut tomber d'un côté du découpage pour son premier exemplaire et de l'autre pour le second. Le modèle verrait alors à l'apprentissage une ligne qu'on lui redemande au test : c'est une fuite. On retire de l'apprentissage les lignes dont une jumelle exacte figure dans le test, puis les copies restantes entre elles. La base de test reste celle du tirage, sans toucher à ses lignes.

In [ ]:
# Votre code :
print("lignes d'apprentissage dont une jumelle est dans le test :", int(jumelles_du_test.sum()))
print("lignes d'apprentissage répétées entre elles :", int((copies & ~jumelles_du_test).sum()))
print("apprentissage après retrait :", len(X_train), "relevés ; test inchangé :", len(X_test), "relevés")

### Étape 6 : construire le modèle de départ

Le modèle de départ est volontairement simple. Un pipeline remplit les cases vides par la médiane de l'apprentissage, convertit le lubrifiant, qui est du texte, en colonnes 0 ou 1, puis ajuste une régression linéaire. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 7 : mesurer sur la base de test

Le modèle prédit les relevés qu'il n'a jamais vus. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en kilonewtons, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.0f} kN")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.0f} kN")
print(f"effort moyen de la base de test : {y_test.mean():.0f} kN")

Lecture du résultat : le R² est d'environ 0,91. L'erreur absolue moyenne est de 50 kN pour un effort moyen de 315 kN, soit 16 %. C'est un bon départ, et il reste de la marge : l'effort croît comme le carré du diamètre, ce qu'une droite ne représente pas. C'est ce chiffre, mesuré sur des relevés que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

## Pour aller plus loin : mesurer le nettoyage, puis améliorer le modèle

Le modèle de départ de l'étape 6 reste le score de référence du défi. Les deux cellules suivantes montrent ce que vaut le nettoyage, puis une démarche complète pour aller plus loin. La première applique le même pipeline au tableau brut, sans nettoyage, sur le même test. La seconde compare par validation croisée, sur l'apprentissage seul, des variantes d'imputation, des variables construites à partir de la formule du refoulement et des modèles non linéaires, puis mesure le meilleur une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 6 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Un relevé brut se regarde avant de se modéliser : extrêmes, unités, doublons, cases vides.
2. Les corrections qui ne dépendent d'aucune autre ligne, conversion d'unités ou code de capteur remplacé par une case vide, peuvent précéder le découpage. Celles qui comparent des lignes, comme les doublons, le suivent.
3. Les cases vides se comblent dans le pipeline, avec les statistiques de l'apprentissage seul, et les catégories se codent en colonnes 0 ou 1.
4. Un nettoyage soigné vaut plus qu'un modèle sophistiqué : sur le tableau brut, le même modèle perd une grande part de son R².
5. Connaître la loi physique aide à construire les variables du modèle ; la validation croisée choisit sans le test.